In [1]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".h5") or f.endswith(".pt"):
            print(os.path.join(root, f))

/kaggle/input/notebooks/fahimratul/prepare-datasheet-for-model/test.h5
/kaggle/input/notebooks/fahimratul/prepare-datasheet-for-model/damage.h5
/kaggle/input/notebooks/fahimratul/train-of-cnn-resnet50/runs/best_resnet.pt
/kaggle/input/notebooks/fahimratul/train-of-cnn-resnet50/runs/last_cnn.pt
/kaggle/input/notebooks/fahimratul/train-of-cnn-resnet50/runs/best_cnn.pt
/kaggle/input/notebooks/fahimratul/train-of-cnn-resnet50/runs/last_resnet.pt


In [2]:
import os
os.makedirs("/kaggle/temp", exist_ok=True)
!cp /kaggle/input/notebooks/fahimratul/prepare-datasheet-for-model/test.h5 /kaggle/temp/test.h5
!ls -lh /kaggle/temp/test.h5

-rw-r--r-- 1 root root 1.8G Jul 25 13:30 /kaggle/temp/test.h5


In [3]:
%%writefile /kaggle/working/06_evaluate_test.py

import argparse
import io
import json
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, precision_recall_fscore_support)
from torch.utils.data import DataLoader, Dataset
from torchvision import models

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
CLASSES = ["no-damage", "minor-damage", "major-damage", "destroyed"]


# ==================================================== Dataset (test)

class H5TestDataset(Dataset):
    """Reads PNG from test.h5 and decodes it. No augmentation."""

    def __init__(self, h5_path, normalize):
        self.h5_path = str(h5_path)
        self.normalize = normalize
        self._h5 = None
        with h5py.File(self.h5_path, "r") as f:
            self.length = f["test"]["label"].shape[0]
            self.labels = f["test"]["label"][:].astype(np.int64)

    def _open(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")
        return self._h5["test"]

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        g = self._open()
        png = g["png"][idx].tobytes()
        arr = np.asarray(Image.open(io.BytesIO(png)).convert("RGB"),
                         dtype=np.float32) / 255.0
        if self.normalize == "imagenet":
            arr = (arr - IMAGENET_MEAN) / IMAGENET_STD
        else:
            arr = (arr - 0.5) / 0.5
        arr = np.ascontiguousarray(arr.transpose(2, 0, 1))
        return torch.from_numpy(arr), int(self.labels[idx])


# ==================================================== model definitions (identical to training)

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(cout)

    def forward(self, x):
        return F.max_pool2d(F.relu(self.bn(self.conv(x))), 2)


class SimpleCNN(nn.Module):
    def __init__(self, num_classes=4, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32), ConvBlock(32, 64), ConvBlock(64, 128),
            ConvBlock(128, 256), ConvBlock(256, 512),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout),
            nn.Linear(512, 256), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.gap(self.features(x)))


def build_resnet(num_classes=4):
    m = models.resnet50(weights=None)      # weights come from the checkpoint
    m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, num_classes))
    return m


@torch.no_grad()
def run_model(model, loader, device):
    model.eval()
    preds, targets = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        logits = model(x)
        preds.append(logits.argmax(1).cpu())
        targets.append(y)
    return torch.cat(preds).numpy(), torch.cat(targets).numpy()


def report_block(name, preds, targets, out_dir):
    print("\n" + "=" * 62)
    print(f"{name} — TEST set (independent, not used in training)")
    print("=" * 62)
    print(classification_report(targets, preds, target_names=CLASSES,
                                digits=4, zero_division=0))
    cm = confusion_matrix(targets, preds)
    print("Confusion matrix (rows=actual, cols=predicted):")
    print(cm)

    macro_f1 = f1_score(targets, preds, average="macro", zero_division=0)
    acc = (preds == targets).mean()
    per_p, per_r, per_f, _ = precision_recall_fscore_support(
        targets, preds, labels=[0, 1, 2, 3], zero_division=0)

    with open(Path(out_dir) / f"test_{name.lower()}.json", "w") as f:
        json.dump({"accuracy": float(acc), "macro_f1": float(macro_f1),
                   "per_class_f1": per_f.tolist(),
                   "confusion_matrix": cm.tolist()}, f, indent=2)
    return {"acc": acc, "macro_f1": macro_f1, "per_f": per_f}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--test_h5", default="/kaggle/temp/test.h5")
    ap.add_argument("--cnn", default="/kaggle/working/runs/best_cnn.pt")
    ap.add_argument("--resnet", default="/kaggle/working/runs/best_resnet.pt")
    ap.add_argument("--out_dir", default="/kaggle/working/test_results")
    ap.add_argument("--batch_size", type=int, default=256)
    ap.add_argument("--workers", type=int, default=2)
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Device: {device}")

    results = {}

    # ----- CNN (simple normalize) -----
    if Path(args.cnn).exists():
        ds = H5TestDataset(args.test_h5, normalize="simple")
        loader = DataLoader(ds, batch_size=args.batch_size, shuffle=False,
                            num_workers=args.workers, pin_memory=True)
        print(f"\nTest patches: {len(ds):,}")
        model = SimpleCNN().to(device)
        ck = torch.load(args.cnn, map_location=device)
        model.load_state_dict(ck["model"])
        preds, targets = run_model(model, loader, device)
        results["CNN"] = report_block("CNN", preds, targets, out_dir)
    else:
        print(f"CNN checkpoint not found: {args.cnn}")

    # ----- ResNet (imagenet normalize) -----
    if Path(args.resnet).exists():
        ds = H5TestDataset(args.test_h5, normalize="imagenet")
        loader = DataLoader(ds, batch_size=args.batch_size, shuffle=False,
                            num_workers=args.workers, pin_memory=True)
        model = build_resnet().to(device)
        ck = torch.load(args.resnet, map_location=device)
        model.load_state_dict(ck["model"])
        preds, targets = run_model(model, loader, device)
        results["ResNet"] = report_block("ResNet", preds, targets, out_dir)
    else:
        print(f"ResNet checkpoint not found: {args.resnet}")

    # ----- side-by-side comparison -----
    if len(results) == 2:
        print("\n" + "=" * 62)
        print("Final comparison (TEST set)")
        print("=" * 62)
        print(f"{'Metric':<18}{'CNN':>12}{'ResNet':>12}")
        print("-" * 42)
        print(f"{'Accuracy':<18}{results['CNN']['acc']:>12.4f}"
              f"{results['ResNet']['acc']:>12.4f}")
        print(f"{'Macro F1':<18}{results['CNN']['macro_f1']:>12.4f}"
              f"{results['ResNet']['macro_f1']:>12.4f}")
        for i, c in enumerate(CLASSES):
            print(f"{c+' F1':<18}{results['CNN']['per_f'][i]:>12.4f}"
                  f"{results['ResNet']['per_f'][i]:>12.4f}")


if __name__ == "__main__":
    main()

Writing /kaggle/working/06_evaluate_test.py


In [4]:
!python /kaggle/working/06_evaluate_test.py \
    --test_h5 /kaggle/temp/test.h5 \
    --cnn /kaggle/input/notebooks/fahimratul/train-of-cnn-resnet50/runs/best_cnn.pt \
    --resnet /kaggle/input/notebooks/fahimratul/train-of-cnn-resnet50/runs/best_resnet.pt \
    --out_dir /kaggle/working/test_results

Device: cuda

Test patches: 106,987

CNN — TEST set (independent, not used in training)
              precision    recall  f1-score   support

   no-damage     0.9743    0.8076    0.8832     79398
minor-damage     0.4297    0.7296    0.5409     11136
major-damage     0.5025    0.7193    0.5917      8455
   destroyed     0.6585    0.8367    0.7370      7998

    accuracy                         0.7947    106987
   macro avg     0.6412    0.7733    0.6882    106987
weighted avg     0.8567    0.7947    0.8136    106987

Confusion matrix (rows=actual, cols=predicted):
[[64122  9228  3486  2562]
 [  800  8125  1867   344]
 [  603  1205  6082   565]
 [  287   351   668  6692]]

ResNet — TEST set (independent, not used in training)
              precision    recall  f1-score   support

   no-damage     0.9574    0.8826    0.9184     79398
minor-damage     0.5257    0.6362    0.5757     11136
major-damage     0.5248    0.6796    0.5923      8455
   destroyed     0.7047    0.8255    0.7603     